# 04 · Paired exact tests and Holm correction

Recompute two-sided exact McNemar p-values from archived discordant counts. Holm correction is applied jointly across all 18 tests.

In [ ]:
from pathlib import Path
import pandas as pd

# Run from the repository root or its notebooks/ directory.
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
candidates = sorted(ROOT.rglob("retained_human_annotations_990.csv"))
if len(candidates) != 1:
    raise FileNotFoundError(
        f"Expected one revised analytical dataset, found {len(candidates)}. "
        "Extract the revised 99-item package into the repository and remove duplicate copies."
    )
DATA = candidates[0].parent
print("Repository:", ROOT)
print("Analysis directory:", DATA)

def load(name):
    return pd.read_csv(DATA / name, encoding="utf-8-sig")


In [ ]:
from scipy.stats import binomtest
import numpy as np
paired = load("qwen_paired_tests_holm.csv")
assert len(paired) == 18 and (paired["n"] == 99).all()
def exact_p(row):
    b, c = int(row["A_only_correct"]), int(row["B_only_correct"])
    return binomtest(b,b+c,0.5,alternative="two-sided").pvalue if b+c else 1.0
p = paired.apply(exact_p,axis=1).to_numpy()
order = np.argsort(p)
adjusted = np.empty(len(p))
adjusted[order] = np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]))
assert np.allclose(p,paired["p_exact_mcnemar"],rtol=1e-6,atol=1e-12)
assert np.allclose(adjusted,paired["p_holm_18_tests"],rtol=1e-6,atol=1e-12)
paired["recomputed_p"] = p
paired["recomputed_holm_p"] = adjusted
paired["reject_05"] = adjusted < 0.05
display(paired)
print("Significant comparisons:",int(paired["reject_05"].sum()))


All nine baseline comparisons remain significant; none of the nine within-condition language comparisons is significant after correction. Nonsignificance does not establish equivalence. These tests concern strict exact match only, not human-rated outcomes.